# Parâmetros do Cellpose no MoNuSeg: diâmetro e limiares

Mede a segmentação do Cellpose (`cpsam`, Cellpose 4.1.1) nas **37 imagens de treino**, com o ground truth gerado pelo
`MonusegDataset`, para escolher os parâmetros do `CellposeStep`. O teste oficial **não** é usado aqui, para que a escolha não
seja feita olhando o conjunto de avaliação.

**Como o Cellpose 4 usa o diâmetro:** a imagem é reescalada pelo fator `30 / diameter` antes da rede (`diameter=None` ou `30`
não reescalam). O diâmetro equivalente mediano dos núcleos, medido nas máscaras do `MonusegDataset`, é de **~22 px** nas
imagens em 40× e de **~12 px** nas três imagens em 20× (`MicronsPerPixel = 0,5005`: `TCGA-HE-7128/7129/7130`). As quatro
imagens sem `MicronsPerPixel` no XML têm núcleos do tamanho das imagens em 40× (~23 px).

| Config. | `diameter` | `flow_threshold` | `min_size` | O que testa |
|---|---|---|---|---|
| `atual` | 30 (fator 1) | 0,2 | 4 | os valores do `CellposeStep` hoje |
| `padrao` | `None` (fator 1) | 0,4 | 15 | os padrões da biblioteca (mesmo fator de escala; só os limiares mudam) |
| `d22` | 22 | 0,2 | 4 | o diâmetro mediano das imagens em 40× para todas |
| `por_imagem` | 22 ou 12 | 0,2 | 4 | 12 nas imagens com `MicronsPerPixel ≥ 0,4` (20×) e 22 nas demais |

`cellprob_threshold = 0` em todas. **Métricas por imagem** (binárias, `segmentation > 0` contra `ground_truth`): Dice, IoU,
precisão, revocação, massa segmentada / massa do GT e número de instâncias / número de núcleos anotados.

**Saída:** `docs/estudo/resultados/cellpose_parametros_treino.csv` (uma linha por imagem e configuração) e
`cellpose_parametros_treino_meta.json` (commit, versões, GPU e configurações). No Colab, a última célula baixa os dois arquivos.

## 0. Setup

**Colab:** ambiente com GPU (o `CellposeStep` exige CUDA). As duas células abaixo clonam ou atualizam o repositório e instalam as dependências.

In [1]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/Pedro-io/cell-fuzzy-seg.git"
BRANCH = "homolog"


def find_repo_root(start="."):
    p = os.path.abspath(start)
    prev = None
    while p != prev:
        if os.path.isdir(os.path.join(p, "src")) and os.path.exists(os.path.join(p, "pyproject.toml")):
            return p
        prev, p = p, os.path.dirname(p)
    return None


if "google.colab" in sys.modules:
    repo_root = os.path.join("/content", "cell-fuzzy-seg")
    if os.path.exists(repo_root):
        # Garante a versão mais recente do branch (o Colab pode ter um clone de outra sessão).
        subprocess.run(["git", "-C", repo_root, "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", repo_root, "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", repo_root, "pull", "--ff-only", "origin", BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, repo_root], check=True)
else:
    repo_root = find_repo_root(os.getcwd())
    if repo_root is None:
        raise RuntimeError("Raiz do repositório não encontrada a partir do diretório atual.")

os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Repositório:", repo_root)

Repositório: /content/cell-fuzzy-seg


In [2]:
if "google.colab" in sys.modules:
    get_ipython().system("python -m pip install -r requirements.txt -q")
else:
    print("Ambiente local: verifique se as dependências do requirements.txt estão instaladas.")

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.4/213.4 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 25.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 84.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 MB 17.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 145.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 35.0 MB/s eta 0:00:00:00:0100:01


## 1. Imports, GPU e versões

In [3]:
import json
import platform
import re
import time

import cellpose
import numpy as np
import pandas as pd
import torch
from scipy.stats import wilcoxon

from src.data.load.monuseg_dataset import MonusegDataset
from src.pipeline.steps.preprocessing.cellpose_step import CellposeStep

assert torch.cuda.is_available(), "Este notebook precisa de GPU (Ambiente de execução → Alterar tipo → GPU)."

COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
SUJO = bool(subprocess.run(["git", "status", "--porcelain", "--", "src", "configs"], capture_output=True, text=True).stdout.strip())
VERSOES = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cellpose": getattr(cellpose, "version", None) or getattr(cellpose, "__version__", "?"),
    "numpy": np.__version__,
    "gpu": torch.cuda.get_device_name(0),
}
print("commit:", COMMIT, "(com mudanças locais em src/ ou configs/!)" if SUJO else "")
print(VERSOES)

commit: 64a624f0743d5d6137fcb1aa5b385378a8c0ac75 
{'python': '3.12.13', 'torch': '2.11.0+cu128', 'cellpose': '4.1.1', 'numpy': '2.0.2', 'gpu': 'Tesla T4'}


## 2. Dados: as 37 imagens de treino

O ground truth vem do `MonusegDataset` (rasterização dos XMLs). A escala de cada imagem vem do atributo `MicronsPerPixel` do XML.

In [4]:
dataset = MonusegDataset(dataset_name="monuseg", config_key="monuseg_training")


def microns_per_pixel(xml_path):
    with open(xml_path, encoding="utf-8") as f:
        m = re.search(r'MicronsPerPixel="([0-9.]*)"', f.read(4096))
    return float(m.group(1)) if m and m.group(1) else None


samples = []
for i in range(len(dataset)):
    s = dataset[i]
    image = s["image"]
    if image.ndim == 3 and image.shape[-1] == 4:
        image = image[..., :3]
    samples.append({
        "id": s["id"],
        "image": image,
        "ground_truth": s["ground_truth"],
        "n_nucleos": int(s["ground_truth_instances"].max()),
        "mpp": microns_per_pixel(s["meta"]["mask_path"]),
    })

print(f"{len(samples)} imagens de treino")
print("em 20× (mpp ≥ 0,4):", [s["id"] for s in samples if s["mpp"] is not None and s["mpp"] >= 0.4])
print("sem MicronsPerPixel:", [s["id"] for s in samples if s["mpp"] is None])

2026-10-02 12:08:49.441 | DEBUG    | src.data.load.monuseg_dataset:_get_file_pairs:183 - Getting file pairs from: image_dir=data_source/MoNuSegTrainingData/Tissue_Images, mask_dir=data_source/MoNuSegTrainingData/Annotations
2026-10-02 12:08:49.443 | DEBUG    | src.data.load.monuseg_dataset:_load_image:148 - Loading image: data_source/MoNuSegTrainingData/Tissue_Images/TCGA-18-5592-01Z-00-DX1.tif
2026-10-02 12:08:50.773 | DEBUG    | src.data.load.monuseg_dataset:_load_image:148 - Loading image: data_source/MoNuSegTrainingData/Tissue_Images/TCGA-21-5784-01Z-00-DX1.tif
2026-10-02 12:08:51.316 | DEBUG    | src.data.load.monuseg_dataset:_load_image:148 - Loading image: data_source/MoNuSegTrainingData/Tissue_Images/TCGA-21-5786-01Z-00-DX1.tif
2026-10-02 12:08:52.063 | DEBUG    | src.data.load.monuseg_dataset:_load_image:148 - Loading image: data_source/MoNuSegTrainingData/Tissue_Images/TCGA-38-6178-01Z-00-DX1.tif
2026-10-02 12:08:53.046 | DEBUG    | src.data.load.monuseg_dataset:_load_image:1

37 imagens de treino
em 20× (mpp ≥ 0,4): ['TCGA-HE-7128-01Z-00-DX1', 'TCGA-HE-7129-01Z-00-DX1', 'TCGA-HE-7130-01Z-00-DX1']
sem MicronsPerPixel: ['TCGA-AY-A8YK-01A-01-TS1', 'TCGA-KB-A93J-01A-01-TS1', 'TCGA-NH-A8F7-01A-01-TS1', 'TCGA-RD-A8N9-01A-01-TS1']


## 3. Configurações

In [ ]:
DIAM_40X = 22.0
DIAM_20X = 12.0


def diametro_por_imagem(sample):
    return DIAM_20X if sample["mpp"] is not None and sample["mpp"] >= 0.4 else DIAM_40X


CONFIGS = {
    "atual":      {"diameter": 30.0, "flow_threshold": 0.2, "min_size": 4},
    "padrao":     {"diameter": None, "flow_threshold": 0.4, "min_size": 15},
    "d22":        {"diameter": DIAM_40X, "flow_threshold": 0.2, "min_size": 4},
    "por_imagem": {"diameter": diametro_por_imagem, "flow_threshold": 0.2, "min_size": 4},
}
CELLPROB_THRESHOLD = 0.0

## 4. Execução

Um único `CellposeStep` (o modelo é carregado uma vez); para cada configuração, os parâmetros do step são trocados antes de cada imagem. Leva alguns minutos numa T4.

In [6]:
def metricas(segmentation, ground_truth):
    p = segmentation > 0
    g = ground_truth > 0
    tp = float((p & g).sum())
    fp = float((p & ~g).sum())
    fn = float((~p & g).sum())
    eps = 1e-8
    return {
        "dice": 2 * tp / (2 * tp + fp + fn + eps),
        "iou": tp / (tp + fp + fn + eps),
        "precisao": tp / (tp + fp + eps),
        "revocacao": tp / (tp + fn + eps),
        "massa_rel": float(p.sum()) / max(float(g.sum()), 1.0),
    }


step = CellposeStep(pretrained_model="cpsam", cellprob_threshold=CELLPROB_THRESHOLD)

linhas = []
t_total = time.time()
for nome, cfg in CONFIGS.items():
    for s in samples:
        diametro = cfg["diameter"](s) if callable(cfg["diameter"]) else cfg["diameter"]
        step.diam_mean = diametro
        step.flow_threshold = cfg["flow_threshold"]
        step.min_size = cfg["min_size"]

        t0 = time.time()
        out = step({"image": s["image"]})
        segundos = time.time() - t0

        seg = out["segmentation"]
        n_inst = int(len(np.unique(seg)) - (1 if (seg == 0).any() else 0))
        linhas.append({
            "config": nome,
            "id": s["id"],
            "mpp": s["mpp"],
            "diameter": diametro,
            "flow_threshold": cfg["flow_threshold"],
            "min_size": cfg["min_size"],
            **metricas(seg, s["ground_truth"]),
            "instancias": n_inst,
            "nucleos_gt": s["n_nucleos"],
            "instancias_rel": n_inst / max(s["n_nucleos"], 1),
            "segundos": segundos,
        })
    print(f"{nome}: ok")

df = pd.DataFrame(linhas)
print(f"{len(df)} linhas em {time.time() - t_total:.0f} s")

100%|██████████| 1.15G/1.15G [00:07<00:00, 176MB/s]    
2026-10-02 12:09:32.289 | INFO     | src.pipeline.steps.preprocessing.cellpose_step:__init__:68 - [CellposeStep] Running on GPU (model=cpsam)
/usr/local/lib/python3.12/dist-packages/cellpose/dynamics.py:524: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  coo = torch.sparse_coo_tensor(pt, torch.ones(pt.shape[1], device=pt.device, dtype=torch.int),


atual: ok
padrao: ok
d22: ok
por_imagem: ok
148 linhas em 2425 s


## 5. Resultados

Média ± desvio padrão sobre as 37 imagens. A comparação com a configuração `atual` é pareada por imagem (diferença média de Dice e teste de Wilcoxon).

In [7]:
pd.set_option("display.width", 200)
METRICAS = ["dice", "iou", "precisao", "revocacao", "massa_rel", "instancias_rel"]

resumo = df.groupby("config", sort=False)[METRICAS].agg(["mean", "std"]).round(3)
print(resumo.to_string())

grupo = np.where(df["mpp"].fillna(0) >= 0.4, "20x", "40x")
print("\nDice médio por grupo de escala:")
print(df.assign(escala=grupo).pivot_table(index="config", columns="escala", values="dice", aggfunc="mean", sort=False).round(3).to_string())

base = df[df.config == "atual"].set_index("id")["dice"]
print("\nComparação pareada com 'atual' (Dice):")
for nome in CONFIGS:
    if nome == "atual":
        continue
    outro = df[df.config == nome].set_index("id")["dice"].loc[base.index]
    dif = outro - base
    p = wilcoxon(outro, base).pvalue if (dif != 0).any() else float("nan")
    print(f"  {nome:11s} diferença média {dif.mean():+.4f} (melhora em {(dif > 0).sum()}/{len(dif)} imagens) | Wilcoxon p = {p:.3g}")

             dice           iou        precisao        revocacao        massa_rel        instancias_rel       
             mean    std   mean    std     mean    std      mean    std      mean    std           mean    std
config                                                                                                        
atual       0.814  0.038  0.688  0.054    0.879  0.038     0.761  0.057     0.867  0.076          0.798  0.099
padrao      0.845  0.030  0.733  0.045    0.867  0.041     0.826  0.048     0.955  0.078          0.922  0.100
d22         0.815  0.039  0.690  0.055    0.872  0.041     0.768  0.058     0.883  0.080          0.803  0.099
por_imagem  0.816  0.038  0.691  0.054    0.871  0.042     0.771  0.056     0.887  0.078          0.806  0.098

Dice médio por grupo de escala:
escala        40x    20x
config                  
atual       0.816  0.790
padrao      0.846  0.836
d22         0.818  0.786
por_imagem  0.818  0.798

Comparação pareada com 'atual' (Dice):


## 6. Salvar e baixar

In [8]:
SAIDA = os.path.join("docs", "estudo", "resultados")
os.makedirs(SAIDA, exist_ok=True)
csv_path = os.path.join(SAIDA, "cellpose_parametros_treino.csv")
meta_path = os.path.join(SAIDA, "cellpose_parametros_treino_meta.json")

df.to_csv(csv_path, index=False)
meta = {
    "commit": COMMIT,
    "mudancas_locais_em_src": SUJO,
    "versoes": VERSOES,
    "conjunto": "treino (37 imagens)",
    "cellprob_threshold": CELLPROB_THRESHOLD,
    "configs": {k: {**v, "diameter": ("12 se MicronsPerPixel >= 0.4, senão 22" if callable(v["diameter"]) else v["diameter"])}
                for k, v in CONFIGS.items()},
    "criado_em": time.strftime("%Y-%m-%d %H:%M:%S"),
}
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print("salvo:", csv_path, "e", meta_path)

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(csv_path)
    files.download(meta_path)

salvo: docs/estudo/resultados/cellpose_parametros_treino.csv e docs/estudo/resultados/cellpose_parametros_treino_meta.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>